# Paper 2 — DL/GNN predictive benchmark (Kaggle GPU arm)

**Run on Kaggle (GPU T4, free).** This notebook is the deep-learning arm of
the pre-registered Stage 10 benchmark: can a GNN predict the
degree-controlled residual R from graph structure **out-of-sample,
subject-grouped** — beyond what degree alone (closed-form) and the numpy
ML arm already achieve? A null result is a strong negative control; a
positive result quantifies learnable structure. **Predictive performance
is not a mechanism claim** (Paper 2 discipline).

Inputs (upload from the repo, or attach as a Kaggle dataset):
- `NODE_FEATURE_MATRIX.parquet` + `continuous_residuals.parquet`
  (already merged per node-subject with `residual_cis`),
- or, for the full-graph GNN: the per-subject cached graphs
  (`03_FEATURE_EXTRACTION/_per_subject_cache/` plus rebuilt adjacency).

Design (mirrors 08_STATISTICS/ml_benchmark.py):
- Split by SUBJECT (5-fold grouped), never by node.
- Target: `residual_cis` (degree already controlled in the target).
- Models: (1) MLP on 12 node features; (2) 2-layer GraphSAGE / GCN on the
  binary thresholded graph with the same features as node attributes.
- Negative control: R permuted within subject.
- Report CV R² (pooled SSE/SST over held-out subjects) ± sd over 3 seeds.

Outputs to save back into the repo under `08_STATISTICS/dl_kaggle/`:
`DL_BENCHMARK_RESULTS.json` + trained-model summary.

In [ ]:
# !pip -q install torch_geometric  # Kaggle GPU image: torch preinstalled
import json, numpy as np, pandas as pd, torch, torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv
SEEDS = [20270927, 20270928, 20270929]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
df = pd.read_parquet('NODE_FEATURE_MATRIX.parquet')  # + merged residual_cis, degree
res = pd.read_parquet('continuous_residuals.parquet')
res = res[res.estimator == 'M1_spline4_raw'][['subject','node','residual_cis']]
df = df.merge(res, on=['subject','node'])
FEATURES = ['strength','betweenness','closeness_d','redundancy','bridge',
            'participation','within_module_z','kcore','eigenvector',
            'pagerank','clustering']
subjects = df.subject.unique()
print(len(df), 'rows;', len(subjects), 'subjects; device', DEVICE)

In [ ]:
def folds(seed=20270927, k=5):
    r = np.random.default_rng(seed)
    return dict(zip(subjects, r.permutation(len(subjects)) % k))

def cv_scores(predict_fn, target='residual_cis', seed=20270927):
    fm = folds(seed)
    sse = sst = 0.0
    for k in range(5):
        te = df[df.subject.map(fm) == k]
        tr = df[df.subject.map(fm) != k]
        pred = predict_fn(tr, te)          # returns np array aligned to te rows
        yte = te[target].to_numpy(float)
        sse += ((yte - pred)**2).sum(); sst += ((yte - yte.mean())**2).sum()
    return 1 - sse/sst

In [ ]:
# ---- MLP arm (per-node features; subject-grouped CV) ----
class MLP(nn.Module):
    def __init__(self, p, h=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(p, h), nn.ReLU(), nn.Dropout(0.2),
                                 nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

def mlp_predict(tr, te, epochs=60, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    cols = ['degree'] + FEATURES
    mu, sd = tr[cols].mean(), tr[cols].std().replace(0,1)
    Xtr = torch.tensor(((tr[cols]-mu)/sd).to_numpy(), dtype=torch.float32, device=DEVICE)
    ytr = torch.tensor(tr.residual_cis.to_numpy(), dtype=torch.float32, device=DEVICE)
    Xte = torch.tensor(((te[cols]-mu)/sd).to_numpy(), dtype=torch.float32, device=DEVICE)
    model = MLP(len(cols)).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    lossf = nn.MSELoss()
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        idx = torch.randperm(len(Xtr), device=DEVICE)
        for i0 in range(0, len(idx), 4096):
            b = idx[i0:i0+4096]
            loss = lossf(model(Xtr[b]), ytr[b])
            loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        return model(Xte).cpu().numpy()

mlp_scores = [cv_scores(lambda tr, te, s=s: mlp_predict(tr, te, seed=s), seed=s)
              for s in SEEDS]
print('MLP CV R2:', mlp_scores, 'mean', np.mean(mlp_scores))

In [ ]:
# ---- GNN arm (GraphSAGE over per-subject thresholded graphs) ----
# Build one big disconnected PyG graph: 456-node subgraphs per subject.
# Edge construction mirrors Stage 6 exactly (top-15% |w| binarization); if
# only the feature parquet is available, edges must be shipped alongside
# (rebuild with 03_FEATURE_EXTRACTION/extract_features.py --dump-edges).
class SAGE(nn.Module):
    def __init__(self, p, h=64):
        super().__init__()
        self.conv1 = SAGEConv(p, h)
        self.conv2 = SAGEConv(h, h)
        self.out = nn.Linear(h, 1)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        return self.out(x).squeeze(-1)

def build_edges(sid):
    """Reconstruct the frozen 15% binary graph for subject sid (fp32 path)."""
    import sys; sys.path.insert(0, '../../02_PREPROCESSING')
    import cache_io
    B = cache_io.threshold_cost(
        cache_io.get_matrix(cache_io.load_key_index(), int(sid),
                            cache_io.PRIMARY_ATLAS, cache_io.PRIMARY_VARIANT), 0.15)
    coo = B.tocoo()
    return torch.tensor(np.vstack([coo.row, coo.col]), dtype=torch.long)

def gnn_predict(tr, te, epochs=50, lr=5e-3, seed=0):
    torch.manual_seed(seed)
    cols = ['degree'] + FEATURES
    all_s = sorted(df.subject.unique())
    idx_map = {s: i for i, s in enumerate(all_s)}
    mu, sd = tr[cols].mean(), tr[cols].std().replace(0,1)
    X = torch.tensor(((df[cols]-mu)/sd).to_numpy(), dtype=torch.float32, device=DEVICE)
    y = torch.tensor(df.residual_cis.to_numpy(), dtype=torch.float32, device=DEVICE)
    train_mask = torch.tensor(df.subject.isin(tr.subject).to_numpy(), device=DEVICE)
    edge_list = [build_edges(s) for s in all_s]
    off = 0; ei = []
    for s, e in zip(all_s, edge_list):
        ei.append(e + off); off += 456
    edge_index = torch.cat(ei, dim=1).to(DEVICE)
    model = SAGE(len(cols), 64).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        out = model(X, edge_index)
        loss = ((out[train_mask] - y[train_mask])**2).mean()
        loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(X, edge_index).cpu().numpy()
    return pred[df.subject.isin(te.subject).to_numpy()]

# gnn_scores = [cv_scores(lambda tr, te, s=s: gnn_predict(tr, te, seed=s), seed=s)
#               for s in SEEDS]   # enable after shipping graph edges to Kaggle
print('GNN arm ready (enable after attaching graph edges)')

In [ ]:
# ---- negative control: R permuted within subject ----
perm = df.copy()
rng = np.random.default_rng(20270927)
for s, d in perm.groupby('subject'):
    perm.loc[d.index, 'residual_cis'] = rng.permutation(d.residual_cis.to_numpy())
# rerun the same CV with target='residual_cis' on `perm` ...

results = {'mlp': {'cv_R2': mlp_scores, 'mean': float(np.mean(mlp_scores))},
           'note': 'subject-grouped CV; predictive benchmark only, not a mechanism claim'}
with open('DL_BENCHMARK_RESULTS.json', 'w') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))